# Fundamentals of Machine Learning — Exercise 10
## Neural Networks: from a Perceptron to a Small MLP



In this exercise, you will implement the essential parts of a perceptron and a small multilayer perceptron (MLP) in NumPy. You will then compare the NumPy network with scikit-learn's `MLPClassifier`, logistic regression, and a random forest.

Most supporting code is prepared. Your work is concentrated in short `TODO` sections and interpretation questions. Complete the notebook in order: later cells depend on earlier solutions.

### Learning outcomes
After completing the exercise, you should be able to:

- explain a neuron and a perceptron as a weighted sum, bias, and activation,
- implement the perceptron update rule and explain why a perceptron cannot solve XOR,
- explain why an MLP needs a hidden layer and a nonlinear activation,
- compute a forward pass and track matrix shapes,
- explain binary cross-entropy, gradient descent, and backpropagation in words,
- complete the parameter-update step of a small NumPy MLP,
- use validation data to compare network configurations without tuning on the test set,
- recognise underfitting and overfitting from the gap between training and test accuracy,
- compare a neural network with simpler library models using the same evaluation protocol.



**Answer checks:** after the final reflection you find collapsed *answer checks* for **all** the 🕵️ questions. Write your own answer first, then compare.

### Symbols used

| Symbol | Meaning |
|---|---|
| 🔮 **Predict** | write your expectation before running the cell |
| 🛠️ **Do** | complete a short `TODO`; the next cell checks it |
| 🕵️ **Think** | explain the result in your own words |

### Important experimental rule

Use the **training set** to fit parameters, the **validation set** to compare configurations, and the **test set only once** for the final evaluation. Repeatedly choosing a model according to test results would contaminate the test set.

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch

from sklearn.datasets import make_moons, load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import BaseEstimator, ClassifierMixin

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
plt.rcParams.update({"figure.figsize": (6, 4), "axes.grid": True, "grid.alpha": .3})


def check(test, ok="Correct.", hint="Not yet."):
    """Self-check: `test` is a function returning True/False (an error counts as a failure)."""
    try:
        passed = bool(test())
    except Exception as e:
        passed = False
        hint = hint + f"  [{type(e).__name__}: {e}]"
    print(("✅ " + ok) if passed else ("❌ " + hint))


# --- helper figures (you do not need to read this part) ---
def fig_neuron():
    fig, ax = plt.subplots(figsize=(8, 3.2)); ax.set_axis_off(); ax.grid(False)
    for i, y in enumerate([0.8, 0.5, 0.2]):
        ax.add_patch(Circle((0.08, y), 0.05, facecolor="#cfe2f3", edgecolor="k"))
        ax.text(0.08, y, f"x{i+1}", ha="center", va="center")
        ax.add_patch(FancyArrowPatch((0.13, y), (0.42, 0.5), arrowstyle="->", mutation_scale=12))
        ax.text(0.26, (y + 0.5) / 2 + 0.04, f"w{i+1}", fontsize=10, color="C3")
    ax.add_patch(Circle((0.5, 0.5), 0.09, facecolor="#fff2cc", edgecolor="k")); ax.text(0.5, 0.5, "Σ + b", ha="center", va="center")
    ax.text(0.5, 0.33, "z = w·x + b", ha="center", fontsize=9)
    ax.add_patch(FancyArrowPatch((0.59, 0.5), (0.68, 0.5), arrowstyle="->", mutation_scale=12))
    ax.add_patch(plt.Rectangle((0.68, 0.42), 0.12, 0.16, facecolor="#d9ead3", edgecolor="k")); ax.text(0.74, 0.5, "f(z)", ha="center", va="center")
    ax.add_patch(FancyArrowPatch((0.8, 0.5), (0.9, 0.5), arrowstyle="->", mutation_scale=12)); ax.text(0.93, 0.5, "a", fontsize=13, va="center")
    for x, t in [(0.08, "inputs"), (0.26, "weights"), (0.5, "sum + bias"), (0.74, "activation")]:
        ax.text(x, 0.02, t, ha="center", fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title("One artificial neuron"); plt.show()


def fig_activations():
    z = np.linspace(-4, 4, 300)
    plt.figure(figsize=(7, 4))
    plt.plot(z, 1 / (1 + np.exp(-z)), label="sigmoid"); plt.plot(z, np.tanh(z), label="tanh"); plt.plot(z, np.clip(z, 0, None), label="ReLU")
    plt.axhline(0, c="k", lw=.8); plt.axvline(0, c="k", lw=.8); plt.ylim(-1.5, 3); plt.xlabel("z"); plt.ylabel("activation(z)"); plt.legend(); plt.title("Common activation functions"); plt.show()


def fig_gd_learning_rates():
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
    ws = np.linspace(-5.5, 5.5, 200)
    for ax, lr, title in zip(axes, [0.05, 0.4, 1.05], ["too small: slow", "good: fast", "too large: diverges"]):
        w_ = 5.0; path = [w_]
        for _ in range(12):
            w_ = w_ - lr * 2 * w_; path.append(w_)
        path = np.array(path)
        ax.plot(ws, ws ** 2, "k-", lw=1); ax.plot(path, path ** 2, "o-", color="C3", ms=4); ax.set_title(f"learning rate {lr}: {title}", fontsize=9); ax.set_xlabel("w"); ax.set_ylim(0, 40)
    axes[0].set_ylabel("loss  L(w) = w²"); plt.tight_layout(); plt.show()


def fig_bce():
    a = np.linspace(0.001, 0.999, 300)
    plt.figure(figsize=(6, 3.4)); plt.plot(a, -np.log(a), label="true label 1:  −log(a)"); plt.plot(a, -np.log(1 - a), label="true label 0:  −log(1 − a)")
    plt.xlabel("predicted probability a"); plt.ylabel("loss"); plt.ylim(0, 5); plt.legend(); plt.title("Binary cross-entropy of one example"); plt.show()


def plot_boundary(predict_proba, X, y, ax=None, title="", grid=150):
    ax = ax or plt.gca()
    x0, x1 = X[:, 0].min() - .5, X[:, 0].max() + .5
    y0, y1 = X[:, 1].min() - .5, X[:, 1].max() + .5
    xx, yy = np.meshgrid(np.linspace(x0, x1, grid), np.linspace(y0, y1, grid))
    pp = predict_proba(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, pp, levels=np.linspace(0, 1, 12), alpha=.35, cmap="coolwarm")
    ax.contour(xx, yy, pp, levels=[.5], linewidths=2, colors="k")
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=18, edgecolor="k", linewidth=.2)
    ax.set_title(title, fontsize=9); ax.grid(False)

---
# 1. Neuron and perceptron 

> 🧠 **Key idea:** a *neuron* is a weighted sum of the inputs plus a bias, passed through a function (the **activation**). The **perceptron** is the neuron with a step activation. You already know the same structure from linear and logistic regression.

$$z=\mathbf w\cdot\mathbf x+b,\qquad a=f(z).$$

The *weights* $\mathbf w$ say how important each input is; the *bias* $b$ shifts the threshold. The perceptron outputs $1$ if $z>0$, else $0$. In 2D, the boundary $z=0$ is a **straight line**.

In [ ]:
fig_neuron()

### 🛠️ Task 1.1 – a neuron computes AND and OR
We start with two tiny logical problems. Choose weights and a bias so that **one neuron** computes **AND** (output 1 only for the input (1, 1)) and then **OR** (output 1 unless the input is (0, 0)). (Hint: both can use the weights `[1, 1]`; only the bias differs. Think: how large must $x_1+x_2$ be to switch the neuron on?)

In [ ]:
X_logic = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_and = np.array([0, 0, 0, 1])
y_or = np.array([0, 1, 1, 1])
y_xor = np.array([0, 1, 1, 0])

def neuron(X, w, b):
    """Perceptron: weighted sum + bias, then the step function."""
    return (X @ w + b > 0).astype(int)

In [ ]:
w_and, b_and = np.array([1.0, 1.0]), None    # TODO: choose the bias for AND
w_or, b_or = np.array([1.0, 1.0]), None      # TODO: choose the bias for OR
print("AND:", neuron(X_logic, w_and, b_and), "  OR:", neuron(X_logic, w_or, b_or))

In [ ]:
check(lambda: (neuron(X_logic, w_and, b_and) == y_and).all(), "AND solved.", "AND must be 1 only for the point (1, 1): what is x1 + x2 there and at the other points? Where must the threshold lie?")
check(lambda: (neuron(X_logic, w_or, b_or) == y_or).all(), "OR solved.", "OR must be 0 only for the point (0, 0): what is x1 + x2 at the other points? Where must the threshold lie?")

### 🛠️ Task 1.2 – how does a perceptron *learn* the weights? (lecture: the perceptron update rule)
Start with $\mathbf w=\mathbf 0$, $b=0$. Go through the points one by one; for every point compute the prediction $\hat y$ and update

$$\mathbf w\leftarrow\mathbf w+\eta\,(y-\hat y)\,\mathbf x,\qquad b\leftarrow b+\eta\,(y-\hat y).$$

If the prediction is correct, $y-\hat y=0$ and nothing changes. If it is wrong, the line moves toward the correct side. One pass over all points is an **epoch**; $\eta$ is the **learning rate**. Fill in the `update` (one line); the rest is given.

In [ ]:
def train_perceptron(X, y, lr=0.2, epochs=25):
    w, b, mistakes = np.zeros(X.shape[1]), 0.0, []
    for _ in range(epochs):
        m = 0
        for xi, yi in zip(X, y):
            pred = int(xi @ w + b > 0)
            update = None                 # TODO: the correction term from the update rule above
            w = w + update * xi
            b = b + update
            m += int(pred != yi)
        mistakes.append(m)
    return w, b, mistakes

w_a, b_a, mist_and = train_perceptron(X_logic, y_and)
w_x, b_x, mist_xor = train_perceptron(X_logic, y_xor)
plt.plot(mist_and, "o-", label="AND"); plt.plot(mist_xor, "o-", label="XOR"); plt.xlabel("epoch"); plt.ylabel("misclassified points"); plt.legend(); plt.show()

<details><summary>💡 Hint</summary>

The rule is $\mathbf w\leftarrow\mathbf w+\eta\,(y-\hat y)\,\mathbf x$. In the code, `lr` is $\eta$, `yi` is $y$ and `pred` is $\hat y$. `update` is the number that multiplies `xi` (for the weights) and is added to `b`: the learning rate times (true label minus prediction).

</details>

In [ ]:
check(lambda: mist_and[-1] == 0 and (neuron(X_logic, w_a, b_a) == y_and).all(), "The perceptron learned AND by itself.", "Use the update rule above: the correction depends on the learning rate and on the difference between the true label and the prediction.")
check(lambda: mist_xor[-1] > 0, "On XOR it still makes mistakes in the last epoch.", "Fill in the update first.")

**Question 1 🕵️.** Complete the sentences. *The perceptron can solve ______ because ______. It cannot solve ______ because ______. Training for more epochs does not help with the second problem because ______.* (Sketch the four points (0,0), (0,1), (1,0), (1,1) and try to separate the XOR classes with one line.)

✍️ **Your answer:** *(double-click this cell and write here)*

---
# 2. Activation functions 
> 🧠 **Key idea:** the hidden activation must be **nonlinear**. Otherwise, several linear layers collapse into one linear transformation. For gradient-based learning, the activation must provide a usable derivative or subgradient. ReLU is not differentiable exactly at zero, but this does not prevent its practical use.

In [ ]:
fig_activations()

| Function | Formula | Range | Typical use |
|---|---|---|---|
| **sigmoid** | $\frac{1}{1+e^{-z}}$ | (0, 1) | the **output** of a binary classifier (a probability) |
| **tanh** | $\tanh z$ | (−1, 1) | hidden layers of small networks (centred at 0) |
| **ReLU** | $\max(0,z)$ | [0, ∞) | the default hidden activation of deep networks |
| softmax | $\frac{e^{z_j}}{\sum_k e^{z_k}}$ | (0, 1), sums to 1 | the output for **more than two classes** (not needed today) |

### 🛠️ Task 2.1
Implement `relu` (it works on a whole NumPy array). `sigmoid` is given.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -50, 50)))

def relu(z):
    return None      # TODO: 0 for negative inputs, z for positive ones (must work on a whole array)
print(relu(np.array([-2.0, 0.0, 3.0])))

<details><summary>💡 Hint</summary>

NumPy has an element-wise maximum of two things: `np.maximum(a, b)`. Compare every element with 0.

</details>

In [ ]:
check(lambda: np.allclose(relu(np.array([-2.0, 0.0, 3.0])), [0, 0, 3]), "ReLU works.", "ReLU is 0 for negative z and z otherwise; open the hint above the check.")

**Question 2 🕵️.** (a) For each of sigmoid, tanh and ReLU: can a *negative* input give a *negative* output? (b) Suppose we replace the hidden activation by the identity $f(z)=z$. Why would the hidden layer then be useless? (Hint: two linear maps in a row, $\mathbf h=\mathbf x W_1$, $\hat y=\mathbf h W_2$.)

✍️ **Your answer:** *(double-click this cell and write here)*

---
# 3. The forward pass of a small MLP

> 🧠 **Key idea:** a **multi-layer perceptron (MLP)** arranges neurons in **layers**: the outputs of one layer are the inputs of the next. The *forward pass* just repeats "multiply by the weights, add the bias, apply the activation". Matrices do **all neurons for all examples at once**, so we must track the **shapes**.

Our network: **2 inputs → 3 hidden neurons (tanh) → 1 output (sigmoid)**, for a batch of 4 examples.

| Object | Meaning | Shape (🔮 predict it first, then check below) |
|---|---|---|
| `X` | 4 examples × 2 features | |
| `W1` | weights input → hidden | |
| `b1` | bias of the hidden neurons | |
| `Z1 = X @ W1 + b1` | | |
| `A1 = tanh(Z1)` | | |
| `W2` | weights hidden → output | |
| `Z2 = A1 @ W2 + b2`, `y_hat = sigmoid(Z2)` | | |

### 🛠️ Task 3.1 – compute the forward pass step by step
The weights are given. Compute `Z1`, `A1`, `Z2` and `y_hat`.

In [ ]:
W1 = np.array([[1.0, -1.0, 0.5], [-0.5, 1.0, 1.0]])
b1 = np.array([[0.0, 0.2, -0.1]])
W2 = np.array([[1.0], [-1.2], [0.8]])
b2 = np.array([[0.1]])

Z1 = None       # TODO: first layer: weighted sums of the inputs plus the bias
A1 = None       # TODO: the hidden activation applied to Z1
Z2 = None       # TODO: second layer: weighted sums of the hidden activations plus the bias
y_hat = None    # TODO: the output activation applied to Z2
print({"X": X_logic.shape, "W1": W1.shape, "b1": b1.shape, "Z1": Z1.shape, "A1": A1.shape, "W2": W2.shape, "Z2": Z2.shape, "y_hat": y_hat.shape})

<details><summary>💡 Hint</summary>

Use the table above. In NumPy, matrix multiplication is the `@` operator: `Z1` is `X_logic` times `W1`, plus `b1`. `A1` is `np.tanh` of `Z1`. `Z2` is `A1` times `W2`, plus `b2`. `y_hat` is `sigmoid` of `Z2`.

</details>

In [ ]:
check(lambda: Z1.shape == (4, 3) and A1.shape == (4, 3) and y_hat.shape == (4, 1) and abs(y_hat[0, 0] - 0.4461) < 1e-3, "Forward pass correct.", "Follow the table of shapes above; y_hat has one probability per example.")

### Debug a deliberate shape error
Matrix shapes are the most common source of errors in neural-network code. The weight matrix below has the **wrong** shape for our network. The cell catches the error so that the notebook can continue. Read the message and the shapes **before** answering Question 3(d).

In [ ]:
BROKEN_W2 = np.array([[1.0], [-1.0]])      # a wrong shape: (2, 1)

try:
    _ = A1 @ BROKEN_W2
except ValueError as error:
    print("Caught error:", error)
    print("\nA1 shape:", A1.shape, "  BROKEN_W2 shape:", BROKEN_W2.shape)

**Question 3 🕵️.** (a) Why does `W1` have the shape `(2, 3)` and `W2` the shape `(3, 1)`? (b) What does one *row* of `A1` contain? (c) Why is the output a probability and not a class label? (d) Look at the error in the cell above: which two dimensions must be equal in `A1 @ W`, and what shape must `W2` have?

✍️ **Your answer:** *(double-click this cell and write here)*

### Package the forward pass as a function

From now on, we use the forward pass as a function. All parameters are represented by the list `[W1, b1, W2, b2]`.


In [ ]:
def forward(X, W1, b1, W2, b2):
    """Forward pass: returns the hidden activations A1 and the output probabilities P."""
    A1 = np.tanh(X @ W1 + b1)
    P = sigmoid(A1 @ W2 + b2)
    return A1, P

check(lambda: np.allclose(forward(X_logic, W1, b1, W2, b2)[1], y_hat), "`forward` gives the same numbers.", "")

---
# 4. Loss, gradient descent, and backpropagation 

> 🧠 **Key idea:** a **loss** measures prediction error, a **gradient** describes how the loss changes with each parameter, gradient descent moves parameters against the gradient, and **backpropagation** computes all gradients efficiently by applying the chain rule from the output toward the input.

For a true label $y\in\{0,1\}$ and predicted probability $p$, binary cross-entropy is

$$\ell(p,y)=-[y\log(p)+(1-y)\log(1-p)].$$

A confident incorrect prediction receives a large loss. The implementation of backpropagation below is supplied: read the shapes and the direction in which information flows; you are not expected to derive every line independently.

In [ ]:
fig_bce()

def bce(y, p):
    p = np.clip(p, 1e-9, 1 - 1e-9); y = np.asarray(y).reshape(-1, 1)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

**Gradient descent:** $w\leftarrow w-\eta\,\partial L/\partial w$. The learning rate $\eta$ is the step size:

In [ ]:
fig_gd_learning_rates()

**Backpropagation (given code, read it).** We go backwards through the network and apply the chain rule at each step. For a sigmoid output with cross-entropy, the first step is simply $dZ_2=(\hat y-y)/n$; every other gradient follows from the previous one by one matrix product and (for the hidden layer) the derivative of tanh, $1-A_1^2$.

| Step | In words | Shape |
|---|---|---|
| `dZ2 = (P - y) / n` | error at the output | `(n, 1)` |
| `dW2 = A1.T @ dZ2` | each hidden neuron's output × the output error | same as `W2` |
| `dA1 = dZ2 @ W2.T` | pass the error back to the hidden layer | `(n, H)` |
| `dZ1 = dA1 * (1 - A1**2)` | multiply by the slope of tanh | `(n, H)` |
| `dW1 = X.T @ dZ1` | each input × the hidden error | same as `W1` |

In [ ]:
def backward(X, y, W1, b1, W2, b2):
    """Gradients of the cross-entropy loss with respect to all parameters."""
    y = np.asarray(y).reshape(-1, 1); n = len(X)
    A1, P = forward(X, W1, b1, W2, b2)
    dZ2 = (P - y) / n
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(axis=0, keepdims=True)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (1 - A1 ** 2)
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(axis=0, keepdims=True)
    return dW1, db1, dW2, db2

grads = backward(X_logic, y_xor, W1, b1, W2, b2)
print({n_: g.shape for n_, g in zip(["dW1", "db1", "dW2", "db2"], grads)})

**Question 4 🕵️.** (a) Why must every gradient have the same shape as the parameter it updates? (b) In words: what does `A1.T @ dZ2` compute?

✍️ **Your answer:** *(double-click this cell and write here)*

---
# 5. Training the MLP: size and learning rate 

> 🧠 **Key idea:** training = repeat {forward pass → loss → backward pass → small step against the gradient}. The size of the network and the learning rate are **hyperparameters** that decide whether it works.

### 🛠️ Task 5.1 – the training loop
Complete the update inside the loop: every parameter gets the gradient descent step. (The rest is given.)

In [ ]:
def init_params(n_in, H, seed=0):
    rng = np.random.default_rng(seed)
    return [rng.normal(0, 0.6, (n_in, H)), np.zeros((1, H)), rng.normal(0, 0.6, (H, 1)), np.zeros((1, 1))]

def train(X, y, H=4, lr=0.5, epochs=2000, seed=0):
    params = init_params(X.shape[1], H, seed)
    losses = []
    for epoch in range(epochs):
        losses.append(bce(y, forward(X, *params)[1]))     # forward pass + loss
        grads = backward(X, y, *params)                   # backward pass
        for i in range(4):
            pass                                          # TODO: one gradient descent step for parameter i
    return params, np.array(losses)

params_xor, loss_xor = train(X_logic, y_xor, H=4, lr=0.5, epochs=3000)
print("XOR probabilities:", forward(X_logic, *params_xor)[1].ravel().round(3), "  final loss:", round(loss_xor[-1], 4))
plt.plot(loss_xor); plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.title("Training on XOR"); plt.show()

<details><summary>💡 Hint</summary>

Gradient descent: $w\leftarrow w-\eta\,\partial L/\partial w$. For each `i`, `params[i]` is the parameter, `grads[i]` its gradient and `lr` the learning rate. Assign the new value back to `params[i]`.

</details>

In [ ]:
check(lambda: loss_xor[-1] < 0.05 and ((forward(X_logic, *params_xor)[1].ravel() > 0.5).astype(int) == y_xor).all(), "The network LEARNED XOR (this time without us choosing the weights).", "Every one of the four parameters must move against its gradient; open the hint above.")

**Question 5 🕵️.** Compare with Part 1: what did the MLP add that the perceptron did not have? Does a low *training* loss alone prove that the network will work on new data?

✍️ **Your answer:** *(double-click this cell and write here)*

### Real nonlinear data: two moons

Two interleaving half-circles cannot be separated by one straight line. We create separate training, validation, and test sets and fit the scaler using the training set only.


In [ ]:
X_m, y_m = make_moons(n_samples=700, noise=0.22, random_state=RANDOM_STATE)

# Hold out the test set first. Do not use it while choosing H or lr.
X_trainval, X_te, y_trainval, y_te = train_test_split(
    X_m, y_m, test_size=0.15, stratify=y_m, random_state=RANDOM_STATE
)
X_tr, X_val, y_tr, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.1765,
    stratify=y_trainval, random_state=RANDOM_STATE
)

scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_val_s = scaler.transform(X_val)

def mlp_scores(H, lr, epochs=1500, seed=0):
    params, losses = train(X_tr_s, y_tr, H=H, lr=lr, epochs=epochs, seed=seed)
    p_val = forward(X_val_s, *params)[1].ravel()
    return params, dict(
        H=H,
        lr=lr,
        train_loss=losses[-1],
        train_acc=np.mean((forward(X_tr_s, *params)[1].ravel() > .5) == y_tr),
        validation_acc=np.mean((p_val > .5) == y_val),
        validation_logloss=log_loss(y_val, p_val),
    )

### 🔮 Predict, then experiment

Below are four configurations defined by the hidden-layer width `H` and learning rate `lr`. Predict which configuration will perform best on the **validation set**, then run the experiment. The purpose is not to find a universally optimal network but to observe how capacity and learning rate affect training.

In [ ]:
pred_best = None    # TODO: your guess, 0..3

configs = [(2, 0.5), (8, 0.5), (8, 0.005), (32, 0.5)]
results = [mlp_scores(H, lr)[1] for H, lr in configs]
table = pd.DataFrame(results)
display(table.round(3))

actual_best = int(table["validation_logloss"].idxmin())
print("Your guess:", pred_best, "  best validation log-loss:", actual_best)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (H, lr) in zip(axes, configs):
    params, result = mlp_scores(H, lr)
    plot_boundary(
        lambda Z: forward(Z, *params)[1].ravel(),
        X_val_s, y_val, ax=ax,
        title=f"H={H}, lr={lr}\nval. log-loss={result['validation_logloss']:.3f}",
    )
plt.tight_layout()
plt.show()

**Question 6 🕵️.**

1. What happens when the learning rate is too small? What could happen if it is too large?
2. Does increasing the number of hidden neurons always improve validation performance?
3. If two configurations perform almost identically, why might you prefer the smaller one?
4. Why must the test set remain unused during this comparison?

### 🛠️ Final evaluation on the untouched test set

Choose `SELECTED_H` and `SELECTED_LR` using the validation table—not the test set. We then refit preprocessing and the selected network on all available training and validation data and evaluate the test set once.

In [ ]:
SELECTED_H = None      # TODO: choose using the validation table, not the test set
SELECTED_LR = None     # TODO: choose from the validation experiment

check(
    lambda: (SELECTED_H, SELECTED_LR) in configs,
    "A tested configuration was selected.",
    "Choose one of the (H, lr) pairs evaluated above (see the list `configs`).",
)

<details><summary>💡 Hint</summary>

Look at the column `validation_logloss` in the table above (lower is better) and take the `H` and `lr` from the same row. Both numbers must come from one row of `configs`.

</details>

In [ ]:
# Run this cell only after selecting the configuration above.
final_scaler = StandardScaler().fit(X_trainval)
X_trainval_s = final_scaler.transform(X_trainval)
X_te_s = final_scaler.transform(X_te)

final_params, final_losses = train(
    X_trainval_s, y_trainval,
    H=SELECTED_H, lr=SELECTED_LR,
    epochs=1500, seed=0,
)
test_prob = forward(X_te_s, *final_params)[1].ravel()

print("Final test accuracy:", round(np.mean((test_prob > .5) == y_te), 3))
print("Final test log-loss:", round(log_loss(y_te, test_prob), 3))

> After examining the test result, do not return to model selection and choose a different configuration. Doing so would turn the test set into another validation set.

✍️ **Your answer:** *(double-click this cell and write here)*

---
# 6. Comparison with other algorithms 

> 🧠 **Key idea:** a from-scratch network is for understanding. In practice you use a library, and you should **always compare a neural network with simpler models**: on small tabular data they are often as good and much cheaper.

We put our network into the scikit-learn interface (given code, you do not need to read it) so that all models can be evaluated in the same way: **5-fold cross-validation**, with the scaler inside the pipeline.

In [ ]:
class NumpyMLP(BaseEstimator, ClassifierMixin):
    """Our network with the scikit-learn interface (binary classification)."""
    def __init__(self, H=8, lr=0.5, epochs=1500):
        self.H, self.lr, self.epochs = H, lr, epochs
    def fit(self, X, y):
        self.classes_ = np.unique(y)
        self.params_, _ = train(X, (y == self.classes_[1]).astype(float), H=self.H, lr=self.lr, epochs=self.epochs)
        return self
    def predict_proba(self, X):
        p = forward(X, *self.params_)[1].ravel(); return np.c_[1 - p, p]
    def predict(self, X):
        return self.classes_[(self.predict_proba(X)[:, 1] > 0.5).astype(int)]

cv5 = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)

def benchmark(models, X, y):
    rows = {}
    for name, model in models.items():
        t0 = time.time()
        s = cross_val_score(make_pipeline(StandardScaler(), model), X, y, cv=cv5)
        rows[name] = dict(accuracy=s.mean(), std=s.std(), seconds=time.time() - t0)
    return pd.DataFrame(rows).T.sort_values("accuracy", ascending=False)

### 🛠️ Task 6.1 – the contestants
Fill in the two missing models.

In [ ]:
models = {
    "logistic regression": LogisticRegression(max_iter=1000),
    "MLP (our NumPy)": NumpyMLP(H=8, lr=0.5, epochs=1500),
    "MLP (scikit-learn)": None,      # TODO: a scikit-learn MLP: one hidden layer of 8 tanh neurons, learning_rate_init=0.01, max_iter=2000, random_state=0
    "random forest": None,           # TODO: a random forest with 200 trees and random_state=0
}

<details><summary>💡 Hint</summary>

The classes are `MLPClassifier(...)` (arguments `hidden_layer_sizes=(8,)`, `activation="tanh"`, `learning_rate_init`, `max_iter`, `random_state`) and `RandomForestClassifier(n_estimators=..., random_state=...)`.

</details>

In [ ]:
check(lambda: all(m is not None for m in models.values()) and len(models) == 4, "Four contestants.", "Fill in both models; open the hint above for the class names.")

### 🔮 Predict
On which data set will the best model be **much better than logistic regression**: the **moons** (curved boundary, 2 features) or the **breast cancer** data (30 features)? Write `"moons"` or `"cancer"`.

In [ ]:
pred_gap = None     # TODO: "moons" or "cancer"
X_mo, y_mo = make_moons(n_samples=600, noise=0.2, random_state=0)
X_bc, y_bc = load_breast_cancer(return_X_y=True)
res_moons, res_cancer = benchmark(models, X_mo, y_mo), benchmark(models, X_bc, y_bc)
print("MOONS:"); print(res_moons.round(3).to_string()); print("\nBREAST CANCER:"); print(res_cancer.round(3).to_string())
gap_m = res_moons["accuracy"].max() - res_moons.loc["logistic regression", "accuracy"]
gap_c = res_cancer["accuracy"].max() - res_cancer.loc["logistic regression", "accuracy"]
actual_gap = "moons" if gap_m > gap_c else "cancer"
print(f"\ngap best - logistic regression: moons {gap_m:.3f}, cancer {gap_c:.3f}   your guess: {pred_gap!r}, actual: {actual_gap!r}")

**Question 7 🕵️.** (a) Which models are on top for each data set, and how big are the differences compared with the `std` (spread over the folds)? (b) Why does the neural network help on the moons but hardly on the breast cancer data? (c) When would you choose a neural network in a project, and when logistic regression or a forest?

✍️ **Your answer:** *(double-click this cell and write here)*

---
# 7. Underfitting and overfitting 

> 🧠 **Key idea (lecture):** *underfitting* = the model is too simple to learn the pattern; *overfitting* = it memorises the training data, including its noise, and fails on new data. Watch the gap between the **training** and the **held-out** accuracy.

We give a small noisy training set (60 points) to a tiny, a medium and a huge network, **without** any regularisation (`alpha=0`). The 1000 other points play the role of held-out data.

In [ ]:
X_s, y_s = make_moons(n_samples=1060, noise=0.4, random_state=3)
X_small, y_small, X_big, y_big = X_s[:60], y_s[:60], X_s[60:], y_s[60:]     # 60 for training, 1000 held out
rows = []
for H in [1, 4, 100]:
    m = MLPClassifier(hidden_layer_sizes=(H,), activation="tanh", alpha=0.0, max_iter=10000, learning_rate_init=0.05, tol=1e-7, n_iter_no_change=200, random_state=0).fit(X_small, y_small)
    train_acc, test_acc = None, None      # TODO: accuracy of the fitted model on the 60 training points, and on the 1000 held-out points
    rows.append((H, train_acc, test_acc, train_acc - test_acc))
overfit = pd.DataFrame(rows, columns=["hidden neurons", "train accuracy", "held-out accuracy", "gap"]).round(3)
display(overfit)

<details><summary>💡 Hint</summary>

Every scikit-learn classifier has a method `.score(X, y)` that returns the accuracy on the given data. The training points are `X_small, y_small`, the held-out points `X_big, y_big`.

</details>

In [ ]:
check(lambda: overfit.loc[2, "train accuracy"] >= 0.95 and overfit.loc[2, "gap"] > overfit.loc[1, "gap"], "The huge network memorised the training points: the biggest gap between train and held-out accuracy.", "Compute the accuracy on the training points and on the held-out points; open the hint above.")

**Question 8 🕵️.** (a) Which row looks like **underfitting** and which like **overfitting**, and how do you see it in the numbers? (b) From the lecture: name three ways to fight overfitting and one to fight underfitting.

✍️ **Your answer:** *(double-click this cell and write here)*

---
# 8. Final reflection 



1. Why can the perceptron solve AND but not XOR?
2. What does the hidden layer add to the model?
3. Which configuration did you select using validation data, and why?
4. Report its final test accuracy and test log-loss.
5. Compare the NumPy MLP with logistic regression and scikit-learn's MLP.
6. Give one limitation of your experiment—for example, dataset size, one random seed, restricted hyperparameter choices, or the use of synthetic data.



### Self-check

- [ ] I can explain a neuron, perceptron, and perceptron update rule.
- [ ] I can explain why a single perceptron cannot solve XOR.
- [ ] I can explain why hidden activations must be nonlinear.
- [ ] I can track matrix shapes through a two-layer network.
- [ ] I can describe loss, gradient descent, and backpropagation in plain language.
- [ ] I completed the NumPy MLP parameter update.
- [ ] I used validation data for model choice and the test set only for final evaluation.
- [ ] I compared the neural network with simpler models under the same protocol.
- [ ] I can recognise underfitting and overfitting from the train/held-out gap.

### Optional extensions

If you want additional practice:

1. Numerically check one gradient using a small positive and negative perturbation.
2. Compare three random seeds for the selected network.
3. Add precision, recall, and F1 to the breast-cancer comparison.
4. Repeat Part 7 with the regularisation `alpha=0.5` and compare the gap.

### Answer checks (compare with your own answers)

<details><summary>Question 1: the perceptron, AND and XOR</summary>

It can solve **AND** (and OR) because the classes can be separated by one straight line. It cannot solve **XOR** because the classes lie on the diagonals of a square: no single line separates them. More epochs do not help because the model cannot represent the solution at all (the number of mistakes keeps jumping instead of reaching 0). The way out is to combine several neurons into a network with a hidden layer.

</details>

<details><summary>Question 2: activations</summary>

(a) sigmoid: no (outputs 0..1); tanh: yes (−1..1); ReLU: no (negative inputs become 0). (b) Two linear layers in a row, $\hat y=(\mathbf xW_1)W_2=\mathbf x(W_1W_2)$, are **one** linear layer with the matrix $W_1W_2$. Without a nonlinear activation, extra layers add nothing and the network could not solve XOR.

</details>

<details><summary>Question 3: shapes of the forward pass</summary>

(a) `W1` connects 2 inputs to 3 hidden neurons (rows = inputs, columns = hidden neurons); `W2` connects 3 hidden neurons to 1 output. (b) One row of `A1` = the activations of all 3 hidden neurons for one example. (c) The sigmoid squeezes the output into (0, 1), which we read as the probability of class 1; a label comes from a threshold (e.g. 0.5). (d) In `A1 @ W`, the number of **columns of `A1` (3)** must equal the number of **rows of `W` (2 in the broken matrix)**, so NumPy raises a `ValueError`. `W2` must have the shape `(3, 1)`.

</details>

<details><summary>Question 4: gradients and shapes</summary>

(a) The update is element by element, $W\leftarrow W-\eta\,dW$, so each weight needs exactly one gradient number. (b) For every connection hidden neuron → output it sums over all examples "how active the hidden neuron was" × "how wrong the output was". A weight changes more if its neuron was active when the output was wrong.

</details>

<details><summary>Question 5: what the MLP added</summary>

A hidden layer with a nonlinear activation (it learns new features), and a smooth loss that gradient descent can follow. A low *training* loss alone proves nothing about new data: it only says that the network fits these points. We need held-out (validation / test) data, see Parts 5 and 7.

</details>

<details><summary>Question 6: learning rate, size, test set</summary>

1. A too small learning rate means tiny steps: after 1500 epochs the training loss is still high (≈ 0.31 against ≈ 0.13 for lr = 0.5) and the validation accuracy is low (≈ 0.83): the network is just not trained yet. A too large rate makes the loss jump around or diverge (see the parabola figure in Part 4).
2. No. With 2 hidden neurons the network is clearly worse (validation log-loss ≈ 0.39); 8 and 32 neurons are close (≈ 0.22 and ≈ 0.20). More neurons help until the capacity is sufficient; beyond that the gain is small and the risk of overfitting grows (Part 7).
3. A smaller network has fewer parameters, trains faster, is more stable and less prone to overfitting, and is easier to explain.
4. If we chose the configuration by the test results, the test set would become part of the selection and the final number would be optimistic (the winner's curse you know from Exercise 9).

</details>

<details><summary>Question 7: comparison with other algorithms</summary>

(a) Moons: the three nonlinear models (both MLPs and the forest) have ≈ 0.97, logistic regression only ≈ 0.88 (about 9 points, far more than the std); the differences among the nonlinear models are within the std (a tie). Breast cancer: all models are around 0.95–0.98, with differences of about one std (logistic regression is among the best). (b) The moons need a curved boundary, which logistic regression cannot produce; logistic regression already models the breast-cancer data well after scaling, so the additional flexibility of the neural network gives little improvement in this experiment (we cannot conclude from these results alone that the classes are linearly separable). (c) A neural network pays off for complex nonlinear patterns and large data (images, text, sound); on small tabular data a simple model or a forest is usually as good, faster, and easier to explain and tune.

</details>

<details><summary>Question 8: underfitting and overfitting</summary>

(a) H = 1: both accuracies are modest (train ≈ 0.85, held-out ≈ 0.81): the model is too simple even for the training data, underfitting. H = 100: training accuracy 1.00, held-out only ≈ 0.74, a gap of ≈ 0.26: overfitting (the network memorised the noise). H = 4 is in between (≈ 0.95 / 0.83). (b) Overfitting: a simpler network (fewer neurons / layers), more data, regularisation (L2 penalty `alpha`, dropout), early stopping. Underfitting: a more complex model (more neurons / layers), better features, longer training.

</details>

<details><summary>Why compare against logistic regression?</summary>

A flexible neural network is not automatically better. Logistic regression is a fast and interpretable baseline; if the neural network does not improve validation performance meaningfully, the simpler model may be preferable.

</details>